# 07 — Trực quan hóa & đóng gói kết quả (M4)

**Mục tiêu:** dựng lại toàn bộ bản đồ/biểu đồ chuẩn hóa từ CSV đã phát hành của M1–M3 (H05, H06). Notebook **không tính lại số liệu** và **không cần dữ liệu thô / nhãn** (`data/`, `outputs/labels/` không có trong repo).

**Đầu vào:** `outputs/metrics/{baseline_metrics,hdbscan_metrics,*_clusters_master_v1,hotspot_tracking}.csv`, `outputs/metrics/eda/pickup_coords.csv`
**Đầu ra:** `metrics_comparison.csv`, `outputs/figures/comparison/*.png` (300 dpi), `outputs/maps/{pickup_demand_heatmap,cluster_map_*,cluster_comparison}.html`, `outputs/m4_artifact_manifest.csv`

**Lưu ý đọc kết quả:** tọa độ là *centroid* (không phải GPS); hotspot là *nhu cầu đón khách*, không phải ùn tắc; noise không được vẽ thành hotspot.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "config" / "experiment.yaml").exists():
    ROOT = ROOT.parent                      # notebook nằm trong notebooks/
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
from src.visualization.build_m4 import build_all
from src.visualization.comparison import merge_metrics
from src.visualization.style import apply_matplotlib_style
apply_matplotlib_style()

## 1. Bảng so sánh thống nhất (H05)
Nối `baseline_metrics.csv` (M2) + `hdbscan_metrics.csv` (M3). Hàm `merge_metrics` dừng nếu schema/sample/n_points khác nhau — không tự sửa giá trị.

In [ ]:
df = merge_metrics(ROOT)
cols = ["time_slice","algorithm","n_points","n_clusters","noise_ratio","coverage","silhouette","davies_bouldin","runtime_fit_sec","weighted_fit","comparable_runtime","run_id"]
df[cols].round(3)

## 2. Dựng lại toàn bộ artifact
`build_all` ghi PNG 300 dpi, HTML, và manifest SHA-256 (truy vết NFR-03). Dùng `dpi=100` khi chạy thử.

In [ ]:
manifest = build_all(ROOT, dpi=300)
manifest[["artifact","source","size_kb"]]

## 3. Xem nhanh
Bản đồ HTML cần Internet để tải nền OSM và thư viện Leaflet; khi mất mạng dùng các PNG dưới đây (bản dự phòng offline).

In [ ]:
fig = ROOT / "outputs" / "figures" / "comparison"
for name in ["runtime_noise_comparison.png", "temporal_hotspot_comparison.png", "algorithm_comparison_WEEKDAY_PM.png"]:
    display(Image(filename=str(fig / name), width=1100))

## 4. Cách đọc đúng (tránh diễn giải sai)
- **Silhouette cao ≠ mô hình tốt hơn.** Chỉ tính trên điểm có nhãn; luôn đọc cùng Coverage. 63% cụm HDBSCAN chỉ gồm 1 tọa độ (bán kính 0) nên Silhouette ≈ 0,98–0,99 bị thổi phồng.
- **Runtime không so trực tiếp:** K-Means/DBSCAN fit *weighted* trên tọa độ duy nhất; HDBSCAN fit trên toàn bộ điểm.
- **Dịch chuyển = 0 m** (>50% cặp) là giới hạn độ phân giải centroid, không phải "đứng yên tuyệt đối".
- Chạy `python scripts/run_qa.py` để xem các cảnh báo còn mở trả về M1/M2/M3.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "run_qa.py")], capture_output=True, text=True)
print(r.stdout[-3500:])